# Auto dataset — KNN modeling (clean)

This notebook contains: loading, EDA, preprocessing, KNN CV, comparison, residuals, and saving artifacts.

In [ ]:
# imports and load
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

colnames = ["mpg","cylinders","displacement","horsepower","weight","acceleration","year","origin","name"]
df = pd.read_csv('Auto.csv', names=colnames, header=0, na_values='?')
df['horsepower'] = pd.to_numeric(df['horsepower'], errors='coerce')
df = df.dropna(subset=['horsepower']).reset_index(drop=True)
df.shape


## EDA

Plot mpg vs displacement/horsepower/weight and boxplot mpg by cylinders.

In [ ]:
import seaborn as sns
sns.set(style='whitegrid')
plt.figure(figsize=(6,4))
sns.scatterplot(x='displacement', y='mpg', data=df)
plt.show()


## Preprocessing and train/test split

In [ ]:
features = ['displacement','horsepower','weight','acceleration','cylinders']
X = df[features]
y = df['mpg']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)


## KNN CV and evaluation

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

param_grid = {'n_neighbors': list(range(1,26,2))}
knn = KNeighborsRegressor()
gs = GridSearchCV(knn, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
gs.fit(X_train_scaled, y_train)
best_k = gs.best_params_['n_neighbors']
best_k, np.sqrt(-gs.best_score_)


## Train weighted KNN and Linear Regression, compare on test

In [ ]:
from sklearn.linear_model import LinearRegression
best_k = int(best_k)
knn_w = KNeighborsRegressor(n_neighbors=best_k, weights='distance')
knn_w.fit(X_train_scaled, y_train)
lr = LinearRegression().fit(X_train_scaled, y_train)

def eval_model(model, X_test_scaled, y_test):
    y_pred = model.predict(X_test_scaled)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    return rmse, r2

print('KNN-weighted:', eval_model(knn_w, X_test_scaled, y_test))
print('LinearRegression:', eval_model(lr, X_test_scaled, y_test))


## Residuals and diagnostics

In [ ]:
import matplotlib.pyplot as plt
y_pred = knn_w.predict(X_test_scaled)
residuals = y_test - y_pred
plt.figure(figsize=(6,4))
sns.histplot(residuals, kde=True)
plt.title('Residuals')
plt.show()


## Save artifacts

In [ ]:
import os
os.makedirs('model_data', exist_ok=True)
joblib.dump(scaler, 'model_data/scaler.joblib')
joblib.dump(knn_w, 'model_data/knn_weighted.joblib')
print('Saved artifacts to model_data/')
